<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/03_sjepa_model_and_pretrain_normal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Build S-JEPA and pretrain on normal gait

Now we build the model and give it its first lesson: learn what ordinary walking looks like. We train only on **normal** clips here. This is phase one of progressive training; ms and pd come in notebook 04.

S-JEPA has three parts, all small transformers:

- a **view encoder** that reads the visible joints of a slightly rotated view,
- a **predictor** that guesses the hidden joints in feature space,
- a **target encoder** that reads the full skeleton and provides the answer. It is a slow moving average of the view encoder, which is what stops the model from cheating by collapsing every skeleton to the same features.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## The two-lane design

The picture below is the whole idea. The top lane makes a prediction from a masked, rotated view. The bottom lane makes the target from the complete skeleton with a slow teacher. The only place they meet is the loss.


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'sjepa_two_lane.svg')))

In [ ]:
from sjepa.config import get_config, describe
from sjepa.models import build_model, pick_device

cfg = get_config()
device = pick_device()
print('device:', device)
print(describe(cfg))
model = build_model(cfg, device=device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable parameters: {n_params/1e6:.2f}M')

## Load the normal-gait windows

We cut each normal sequence into overlapping windows. Each window is one training example.


In [ ]:
from sjepa.data import load_index, SequenceWindowDataset

records = load_index(KEYPOINTS_DIR)
normal_records = [r for r in records if r.label == 'normal']
ds_normal = SequenceWindowDataset(normal_records, cfg.window_frames, cfg.window_stride)
print(f'{len(normal_records)} normal videos -> {len(ds_normal)} training windows')

## Pretrain

We run the two-lane objective: predict the hidden joints' features, match them to the slow teacher's features with a centered, sharpened cross-entropy, and nudge the teacher along with an exponential moving average. Watch the loss fall.


In [ ]:
from sjepa.train import train_sjepa, save_checkpoint

state = train_sjepa(model, ds_normal, cfg, epochs=cfg.pretrain_epochs,
                    device=device, log_every=max(1, cfg.pretrain_epochs))
save_checkpoint(ARTIFACT_DIR / 'sjepa_pretrain_normal.pt', model, cfg,
                extra={'stage': 'pretrain_normal'})
print('final loss (mean of last 5 steps):',
      round(sum(state.losses[-5:]) / min(5, len(state.losses)), 4))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.plot(state.losses, color='#dd6b20')
plt.xlabel('training step'); plt.ylabel('loss'); plt.title('S-JEPA pretraining on normal gait')
plt.tight_layout(); plt.show()

### Sanity checks

Two things to confirm. First, the loss went down. Second, the teacher and the view encoder are not identical, which is the sign that the anti-collapse machinery is working.


In [ ]:
import torch
early = sum(state.losses[:5]) / 5
late = sum(state.losses[-5:]) / 5
print(f'loss {early:.3f} -> {late:.3f}')
assert late <= early + 1e-3, 'loss did not decrease'
diff = sum(torch.norm(t - v).item() for t, v in
           zip(model.target_encoder.parameters(), model.view_encoder.parameters()))
print('teacher vs student weight distance:', round(diff, 3))
assert diff > 0, 'teacher collapsed onto student'
print('Pretraining looks healthy. On to progressive fine-tuning.')